In [29]:
from pyspark. sql import SparkSession
from pyspark. sql. functions import col, udf
from pyspark. sql.types import StringType, IntegerType
import time
from pyspark import StorageLevel

In [30]:
spark = SparkSession.builder.appName("AdvanceSparkApp").getOrCreate()


In [31]:
df = spark.range(0, 10000000).withColumn("value", col("id") % 1000)

In [32]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [33]:
print("Before Partition:" ,df.rdd.getNumPartitions())

Before Partition: 2


In [34]:
df_repartitioned = df.repartition(20)

In [35]:
print("After Partitions:", df_repartitioned.rdd.getNumPartitions())


[Stage 35:>                                                         (0 + 2) / 2]

After Partitions: 20


[Stage 35:=============================>                            (1 + 1) / 2]

In [36]:
df_coalesced = df_repartitioned.coalesce(50)

In [37]:
print("After Coalesced:" , df_coalesced.rdd.getNumPartitions())

[Stage 36:>                                                         (0 + 2) / 2]

After Coalesced: 20


[Stage 36:=============================>                            (1 + 1) / 2]

In [38]:
df_repartitioned.write.mode("overwrite").csv("output/spark-advance", header=True)


In [39]:
df_coalesced.write.mode("overwrite").csv("output/spark", header=True)

In [40]:
#optimization and caching

In [41]:
optimized_df = df.filter(col("value")>500).filter(col("id")<500000).select("id","value")

In [42]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#377L, (id#377L % 1000) AS value#379L]
+- *(1) Filter (((id#377L % 1000) > 500) AND (id#377L < 500000))
   +- *(1) Range (0, 10000000, step=1, splits=2)




In [43]:
start_time = time.time()
count_uncached = optimized_df.count() # Action triggers DAG
end_time = time.time()
print(f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time,4)}seconds")

1. Optimized Execution | Count: 249500 | Time: 0.288seconds


In [44]:
optimized_df.cache()

DataFrame[id: bigint, value: bigint]

In [45]:
start_time = time.time()
count_uncached = optimized_df.count() # Action triggers DAG
end_time = time.time()
print(f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time,4)}seconds")

1. Optimized Execution | Count: 249500 | Time: 0.5812seconds


In [46]:
#persist command

In [47]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [48]:
optimized_df.persist(StorageLevel.MEMORY_ONLY)

DataFrame[id: bigint, value: bigint]

In [49]:
optimized_df.persist(StorageLevel.MEMORY_AND_DISK)

26/06/13 06:20:08 WARN CacheManager: Asked to cache already cached data.


DataFrame[id: bigint, value: bigint]

In [51]:
print("Data Persisted")
df.count()
df.show(10)

Data Persisted
+---+-----+
| id|value|
+---+-----+
|  0|    0|
|  1|    1|
|  2|    2|
|  3|    3|
|  4|    4|
|  5|    5|
|  6|    6|
|  7|    7|
|  8|    8|
|  9|    9|
+---+-----+
only showing top 10 rows



In [50]:
optimized_df.persist(StorageLevel.DISK_ONLY)

26/06/13 06:24:45 WARN CacheManager: Asked to cache already cached data.


DataFrame[id: bigint, value: bigint]

In [53]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [54]:
#User Defined Functions

In [55]:
data = [("Alice",25), ("Bob", 17), ("Charlie", 35), ("David", 15)]
df1 = spark.createDataFrame(data, ["Name", "Age"])

In [56]:
df1.show()

[Stage 54:>                                                         (0 + 1) / 1]

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 17|
|Charlie| 35|
|  David| 15|
+-------+---+



In [63]:
def categorize_age(age):
    if age >= 18:
        return "Adult"
    return "Minor"

In [65]:
age_category_udf = udf(categorize_age, StringType())

In [66]:
df_method1 = df1.withColumn("Category", age_category_udf(col("Age")))
print("Method 1: Standard UDF via DataFrame API")
df_method1.show()

Method 1: Standard UDF via DataFrame API


+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
|  David| 15|   Minor|
+-------+---+--------+



In [67]:
#for sql


In [68]:
spark.udf.register("sql_categorize_age", categorize_age, StringType())

df1.createOrReplaceTempView("people")

In [77]:
sql_df = spark.sql("""
    SELECT
        Name,
        Age,
        sql_categorize_age(Age) AS Category
    FROM people
""")

sql_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
|  David| 15|   Minor|
+-------+---+--------+



In [78]:
def df_candrive(age):
    if age>18:
        return "canDrive"
    return "cannotDrive"


In [81]:
df = spark.createDataFrame(
    [("Ruchika", 21), ("Priya", 16)],
    ["name", "age"]
)

In [82]:
df.createOrReplaceTempView("person")

In [83]:
sql_df = spark.sql("""
    SELECT
        age,
        CASE
            WHEN age >= 18 THEN 'canDrive'
            ELSE 'cannotDrive'
        END AS driving_status
    FROM person
""")

sql_df.show()

+---+--------------+
|age|driving_status|
+---+--------------+
| 21|      canDrive|
| 16|   cannotDrive|
+---+--------------+



In [84]:
spark.catalog.listTables()

[Table(name='people', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='person', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]